In [ ]:
# run this notebook after 14_analyze_siblings_standard_QC_ibd2_R
# use this notebook to prepare files necessary for allele frequency calculation
# after this, run 16_calculate_af_python

In [ ]:
library(data.table)
library(dplyr)

In [ ]:
sibs = fread("./relatedness/sibs.txt")
trios = fread("./relatedness/trios.txt")
twins = fread("./relatedness/twin_dup.txt")
relatedness = fread("./king_files/relatedness.tsv") # from aou 
flagged = fread("./flagged_samples.tsv") # from aou

In [ ]:
all_samples = c(sibs$IID1, sibs$IID2,trios$par1,trios$par2,trios$off,twins$IID1,twins$IID2)
all_samples = unique(all_samples)

In [ ]:
# anyone who is >= 0.177 to any sib trio twin 
remove = c()
relatedness = relatedness %>% filter(kin >= 0.177)
for (i in 1:nrow(relatedness)){
    if (relatedness$i.s[i] %in% all_samples){
        remove = c(remove, relatedness$j.s[i])
    } else if (relatedness$j.s[i] %in% all_samples){
        remove = c(remove, relatedness$i.s[i])
    }
}

In [ ]:
remove = unique(c(remove, all_samples))

In [ ]:
fwrite(list(remove),"./samples_to_remove.txt", sep = "\t", quote = FALSE, row.names=FALSE)

In [ ]:
# all difs 
twindifs = fread("./aou_twin_differences_giab_removed.txt")
triodifs = fread("./trios_snps_after_standard_QC_GIAB.txt")
sibdifs = fread("./sibs_snps_QC_GIAB.txt")

In [ ]:
triodifs$locus = paste("chr", triodifs$chr, ":", triodifs$pos, sep = "")
sibdifs$locus = paste("chr", sibdifs$chr, ":", sibdifs$pos, sep = "")
alldifs = twindifs[,c("locus", "alleles")]
alldifs = rbind(alldifs, sibdifs[,c("locus", "alleles")])
alldifs = rbind(alldifs, triodifs[,c("locus", "alleles")])

In [ ]:
fwrite(alldifs,"./all_snps_QC_for_AF.txt", sep = "\t", quote = FALSE, row.names=FALSE)